# Bobby Carrot RL — Colab GPU Training

Runs the full pipeline on Google Colab:
1. Setup (Drive, repo, deps)
2. Copy expert data from Drive
3. BC pre-training (joint → per-level)
4. Per-level PPO training
5. Benchmark + download

**Hyperparameter notes (updated):**
- `--lr 3e-5` — fine-tuning rate that allows meaningful PPO updates from BC warm-start
- `--entropy 0.05` — enough exploration to escape BC determinism; decays to 0.01
- `--kl-coef 0.3` — strong BC anchor early, annealed to 0.03 over training so policy can explore
- `--explore-eps 0.10` — 10% forced random actions to break deterministic BC lock-in
- Hard levels (no BC data): `--timesteps 300000` gives extra exploration budget

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# ── Configure these paths to match your Drive layout ──────────────────────
DRIVE_ROOT  = '/content/drive/MyDrive/Bobby_Carrot_RL'
EXPERT_DIR  = f'{DRIVE_ROOT}/expert_data'
BC_DIR      = f'{DRIVE_ROOT}/checkpoints/bc'
CKPT_DIR    = f'{DRIVE_ROOT}/checkpoints/per_level'
# ──────────────────────────────────────────────────────────────────────────

os.makedirs(DRIVE_ROOT,  exist_ok=True)
os.makedirs(EXPERT_DIR,  exist_ok=True)
os.makedirs(BC_DIR,      exist_ok=True)
os.makedirs(CKPT_DIR,    exist_ok=True)
print('Drive paths ready:', DRIVE_ROOT)

## 2. Clone Repository

In [ ]:
# Update this URL to match your GitHub repo
REPO_URL = 'https://github.com/Charan20510/RL_Game_Project.git'

!git clone {REPO_URL} /content/RL_Game_Project
%cd /content/RL_Game_Project

## 3. Install Dependencies

In [ ]:
!pip install -r requirements.txt -q
# If requirements.txt is missing, install manually:
# !pip install pygame torch numpy

## 4. Copy Expert Data from Drive

Run expert dataset generation **locally** first:
```bash
python -m Bobby_Carrot.rl_models.dataset_gen --levels 1-30 --output-dir expert_data
```
Then upload `expert_data/` to `DRIVE_ROOT` on Drive before running this cell.

In [ ]:
!mkdir -p expert_data
!cp -r {EXPERT_DIR}/*.npz expert_data/ 2>/dev/null && echo 'Expert data copied' || echo 'No .npz files found in Drive — run dataset_gen locally first'

## 5. BC Pre-Training

**Option A (recommended):** Run BC here on Colab GPU — fast (~2 min).

**Option B:** Copy pre-trained BC checkpoints from Drive (if already trained locally).

BC now uses `label_smoothing=0.1` to prevent entropy collapse, which was the root cause of the `coll=78.9% constant` lock-in seen in earlier training runs.

In [ ]:
import os
has_joint_bc = os.path.exists(f'{BC_DIR}/joint_bc.pt')
print('joint_bc.pt on Drive:', has_joint_bc)
print('Run Option A (train here) or Option B (copy from Drive) below.')

In [ ]:
# ── Option A: Train BC on Colab (~2 min on GPU) ───────────────────────────
!mkdir -p checkpoints/bc

# Step A1: Joint BC on all available expert datasets
!python -m Bobby_Carrot.rl_models.bc_pretrain \
    --data-dir expert_data \
    --ckpt-dir checkpoints/bc \
    --joint --joint-epochs 100 --device cuda

# Step A2: Per-level fine-tune starting from joint_bc.pt
!python -m Bobby_Carrot.rl_models.bc_pretrain \
    --data-dir expert_data \
    --ckpt-dir checkpoints/bc \
    --resume checkpoints/bc/joint_bc.pt \
    --epochs 80 --device cuda

# Save BC checkpoints to Drive so they survive session restarts
!cp -r checkpoints/bc/* {BC_DIR}/
print('BC checkpoints saved to Drive.')

In [ ]:
# ── Option B: Copy BC checkpoints from Drive (skip if running Option A) ───
!mkdir -p checkpoints/bc
!cp -r {BC_DIR}/* checkpoints/bc/
!ls checkpoints/bc/

## 6. Per-Level PPO Training

Checkpoints are written **directly to Drive** (`CKPT_DIR`) so they persist across session disconnects.

**Level groups:**
- **Easy** (have expert data, 200k steps): 1–6, 8–14, 20, 25–26, 28–30
- **Hard** (no expert data, 300k steps): 7, 15–19, 21–24, 27 — these rely entirely on RL exploration

Use the **batch cells** to train all levels at once, or scroll to the individual level cells to train/retrain specific levels.

### 6a. Batch Training (all levels in one run)

In [ ]:
# Easy levels — have BC expert data, 200k steps each
!python -m Bobby_Carrot.rl_models.per_level_train \
    --levels 1-6,8-14,20,25-26,28-30 \
    --bc-dir checkpoints/bc \
    --ckpt-dir {CKPT_DIR} \
    --timesteps 200000 \
    --device cuda

In [ ]:
# Hard levels — no expert data (falls back to joint_bc.pt), need 300k steps
!python -m Bobby_Carrot.rl_models.per_level_train \
    --levels 7,15-19,21-24,27 \
    --bc-dir checkpoints/bc \
    --ckpt-dir {CKPT_DIR} \
    --timesteps 300000 \
    --device cuda

### 6b. Individual Level Cells

Run these to train or retrain a specific level without touching others.
All cells use the corrected hyperparameters.

In [ ]:
# Level 1
!python -m Bobby_Carrot.rl_models.per_level_train \
    --levels 1 --bc-dir checkpoints/bc --ckpt-dir {CKPT_DIR} \
    --timesteps 200000 --device cuda

In [ ]:
# Level 2
!python -m Bobby_Carrot.rl_models.per_level_train \
    --levels 2 --bc-dir checkpoints/bc --ckpt-dir {CKPT_DIR} \
    --timesteps 200000 --device cuda

In [ ]:
# Level 3
!python -m Bobby_Carrot.rl_models.per_level_train \
    --levels 3 --bc-dir checkpoints/bc --ckpt-dir {CKPT_DIR} \
    --timesteps 200000 --device cuda

In [ ]:
# Level 4
!python -m Bobby_Carrot.rl_models.per_level_train \
    --levels 4 --bc-dir checkpoints/bc --ckpt-dir {CKPT_DIR} \
    --timesteps 200000 --device cuda

In [ ]:
# Level 5 (partial expert data — KL annealing + epsilon-greedy handle the gap)
!python -m Bobby_Carrot.rl_models.per_level_train \
    --levels 5 --bc-dir checkpoints/bc --ckpt-dir {CKPT_DIR} \
    --timesteps 300000 --device cuda

In [ ]:
# Level 6
!python -m Bobby_Carrot.rl_models.per_level_train \
    --levels 6 --bc-dir checkpoints/bc --ckpt-dir {CKPT_DIR} \
    --timesteps 200000 --device cuda

In [ ]:
# Level 7 (no expert data — uses joint_bc.pt, needs extra steps)
!python -m Bobby_Carrot.rl_models.per_level_train \
    --levels 7 --bc-dir checkpoints/bc --ckpt-dir {CKPT_DIR} \
    --timesteps 300000 --device cuda

In [ ]:
# Level 8
!python -m Bobby_Carrot.rl_models.per_level_train \
    --levels 8 --bc-dir checkpoints/bc --ckpt-dir {CKPT_DIR} \
    --timesteps 200000 --device cuda

In [ ]:
# Level 9
!python -m Bobby_Carrot.rl_models.per_level_train \
    --levels 9 --bc-dir checkpoints/bc --ckpt-dir {CKPT_DIR} \
    --timesteps 200000 --device cuda

In [ ]:
# Level 10
!python -m Bobby_Carrot.rl_models.per_level_train \
    --levels 10 --bc-dir checkpoints/bc --ckpt-dir {CKPT_DIR} \
    --timesteps 200000 --device cuda

In [ ]:
# Level 11
!python -m Bobby_Carrot.rl_models.per_level_train \
    --levels 11 --bc-dir checkpoints/bc --ckpt-dir {CKPT_DIR} \
    --timesteps 200000 --device cuda

In [ ]:
# Level 12
!python -m Bobby_Carrot.rl_models.per_level_train \
    --levels 12 --bc-dir checkpoints/bc --ckpt-dir {CKPT_DIR} \
    --timesteps 200000 --device cuda

In [ ]:
# Level 13
!python -m Bobby_Carrot.rl_models.per_level_train \
    --levels 13 --bc-dir checkpoints/bc --ckpt-dir {CKPT_DIR} \
    --timesteps 200000 --device cuda

In [ ]:
# Level 14
!python -m Bobby_Carrot.rl_models.per_level_train \
    --levels 14 --bc-dir checkpoints/bc --ckpt-dir {CKPT_DIR} \
    --timesteps 200000 --device cuda

In [ ]:
# Level 15 (no expert data — arrow/conveyor maze)
!python -m Bobby_Carrot.rl_models.per_level_train \
    --levels 15 --bc-dir checkpoints/bc --ckpt-dir {CKPT_DIR} \
    --timesteps 300000 --device cuda

In [ ]:
# Level 16 (no expert data)
!python -m Bobby_Carrot.rl_models.per_level_train \
    --levels 16 --bc-dir checkpoints/bc --ckpt-dir {CKPT_DIR} \
    --timesteps 300000 --device cuda

In [ ]:
# Level 17 (no expert data)
!python -m Bobby_Carrot.rl_models.per_level_train \
    --levels 17 --bc-dir checkpoints/bc --ckpt-dir {CKPT_DIR} \
    --timesteps 300000 --device cuda

In [ ]:
# Level 18 (no expert data)
!python -m Bobby_Carrot.rl_models.per_level_train \
    --levels 18 --bc-dir checkpoints/bc --ckpt-dir {CKPT_DIR} \
    --timesteps 300000 --device cuda

In [ ]:
# Level 19 (no expert data)
!python -m Bobby_Carrot.rl_models.per_level_train \
    --levels 19 --bc-dir checkpoints/bc --ckpt-dir {CKPT_DIR} \
    --timesteps 300000 --device cuda

In [ ]:
# Level 20
!python -m Bobby_Carrot.rl_models.per_level_train \
    --levels 20 --bc-dir checkpoints/bc --ckpt-dir {CKPT_DIR} \
    --timesteps 200000 --device cuda

In [ ]:
# Level 21 (no expert data)
!python -m Bobby_Carrot.rl_models.per_level_train \
    --levels 21 --bc-dir checkpoints/bc --ckpt-dir {CKPT_DIR} \
    --timesteps 300000 --device cuda

In [ ]:
# Level 22 (no expert data)
!python -m Bobby_Carrot.rl_models.per_level_train \
    --levels 22 --bc-dir checkpoints/bc --ckpt-dir {CKPT_DIR} \
    --timesteps 300000 --device cuda

In [ ]:
# Level 23 (no expert data)
!python -m Bobby_Carrot.rl_models.per_level_train \
    --levels 23 --bc-dir checkpoints/bc --ckpt-dir {CKPT_DIR} \
    --timesteps 300000 --device cuda

In [ ]:
# Level 24 (no expert data)
!python -m Bobby_Carrot.rl_models.per_level_train \
    --levels 24 --bc-dir checkpoints/bc --ckpt-dir {CKPT_DIR} \
    --timesteps 300000 --device cuda

In [ ]:
# Level 25
!python -m Bobby_Carrot.rl_models.per_level_train \
    --levels 25 --bc-dir checkpoints/bc --ckpt-dir {CKPT_DIR} \
    --timesteps 200000 --device cuda

In [ ]:
# Level 26
!python -m Bobby_Carrot.rl_models.per_level_train \
    --levels 26 --bc-dir checkpoints/bc --ckpt-dir {CKPT_DIR} \
    --timesteps 200000 --device cuda

In [ ]:
# Level 27 (no expert data)
!python -m Bobby_Carrot.rl_models.per_level_train \
    --levels 27 --bc-dir checkpoints/bc --ckpt-dir {CKPT_DIR} \
    --timesteps 300000 --device cuda

In [ ]:
# Level 28
!python -m Bobby_Carrot.rl_models.per_level_train \
    --levels 28 --bc-dir checkpoints/bc --ckpt-dir {CKPT_DIR} \
    --timesteps 200000 --device cuda

In [ ]:
# Level 29
!python -m Bobby_Carrot.rl_models.per_level_train \
    --levels 29 --bc-dir checkpoints/bc --ckpt-dir {CKPT_DIR} \
    --timesteps 200000 --device cuda

In [ ]:
# Level 30
!python -m Bobby_Carrot.rl_models.per_level_train \
    --levels 30 --bc-dir checkpoints/bc --ckpt-dir {CKPT_DIR} \
    --timesteps 200000 --device cuda

## 7. Benchmark All Checkpoints

In [ ]:
!python -m Bobby_Carrot.rl_models.play_agent \
    --checkpoint {CKPT_DIR} \
    --benchmark \
    --episodes 10 \
    --no-render \
    --device cuda

## 8. Download Checkpoints

Checkpoints were written directly to `CKPT_DIR` on Drive, so they are already saved.
Use the cell below to zip them for easier download if needed.

In [ ]:
import os
zip_path = f'{DRIVE_ROOT}/per_level_checkpoints.zip'
!zip -r {zip_path} {CKPT_DIR}/*.pt
print(f'Zipped to: {zip_path}')

# To download directly to your browser:
# from google.colab import files
# files.download(zip_path)